# BAW Notebook 00a: Multi-Structure SASA Analysis

**Purpose:** Build a consensus solvent accessibility profile for a target protein
by aligning multiple PDB structures to the canonical UniProt sequence and
averaging SASA values across all structures.

**Why this matters:** A single crystal structure gives a SASA snapshot of one
conformation. Residues that appear buried in one structure (e.g. Tyr119 and
Trp312 in 1S0I which are occluded by the substrate) may be genuinely accessible
in other conformations. Multi-structure averaging gives a reliable picture of
which residues are accessible across the conformational ensemble.

**Input:**
- UniProt accession number (fetched automatically from UniProt REST API)
- Two or more PDB files of the same protein (different conformations, mutants,
  ligand states)

**Output:**
- `{target_label}_consensus_sasa.json` — per-residue consensus SASA profile
  in canonical UniProt numbering, ready for Notebook 01

**Notebook 01 integration:** Notebook 01 automatically loads this file if it
exists in the data directory and uses the consensus SASA instead of
single-structure SASA for candidate pool selection.


In [1]:
# Cell 1: Imports

import os
import sys
import json
import subprocess
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from collections import defaultdict

from Bio.PDB import PDBParser, PPBuilder
from Bio.PDB.SASA import ShrakeRupley
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

warnings.filterwarnings('ignore')

# --------------------------------------------------------------------------
# Shared plotting palette -- kept consistent with Notebook 01
# --------------------------------------------------------------------------
CATEGORY_COLORS = {
    'catalytic':   '#C0392B',   # deep red
    'hydrophobic': '#E67E22',   # orange
    'surface':     '#2980B9',   # blue
    'polar':       '#27AE60',   # green
    'unknown':     '#95A5A6',   # grey
}
SASA_COLORMAP = 'RdYlGn'  # red=buried, green=exposed
CATEGORY_SASA = {
    'A_consistently_exposed': '#2ECC71',
    'B_mostly_exposed':       '#F39C12',
    'C_variable':             '#E74C3C',
    'D_mostly_buried':        '#95A5A6',
    'missing':                '#BDC3C7',
}

# ── PowerShell popup helpers ────────────────────────────────────────────────
def ps_browse_files_multi(title="Select PDB files"):
    """Open a Windows multi-select file dialog via PowerShell."""
    script = f"""
Add-Type -AssemblyName System.Windows.Forms
$d = New-Object System.Windows.Forms.OpenFileDialog
$d.Title = "{title}"
$d.Filter = "PDB files|*.pdb|All files|*.*"
$d.Multiselect = $true
$d.ShowDialog() | Out-Null
$d.FileNames -join "|"
"""
    r = subprocess.run(["powershell.exe", "-NoProfile", "-Command", script],
                       capture_output=True, text=True)
    raw = r.stdout.strip()
    if not raw:
        return []
    paths = []
    for p in raw.split("|"):
        p = p.strip()
        if not p:
            continue
        w = subprocess.run(["wslpath", p], capture_output=True, text=True)
        wsl = w.stdout.strip()
        if wsl:
            paths.append(wsl)
    return paths

def ps_browse_folder(title="Select output folder"):
    script = f"""
Add-Type -AssemblyName System.Windows.Forms
$d = New-Object System.Windows.Forms.FolderBrowserDialog
$d.Description = "{title}"
$d.ShowNewFolderButton = $true
$d.ShowDialog() | Out-Null
Write-Output $d.SelectedPath
"""
    r = subprocess.run(["powershell.exe", "-NoProfile", "-Command", script],
                       capture_output=True, text=True)
    p = r.stdout.strip()
    if not p:
        return None
    w = subprocess.run(["wslpath", p], capture_output=True, text=True)
    return w.stdout.strip() or None

print("Imports loaded.")


Imports loaded.


In [2]:
# Cell 2: Configuration

W = dict(style={'description_width': '160px'})

target_name_w = widgets.Text(
    value='', description='Target name:',
    placeholder='e.g. Trans-sialidase',
    layout=widgets.Layout(width='400px'), **W)

target_label_w = widgets.Text(
    value='', description='File label:',
    placeholder='e.g. ts',
    layout=widgets.Layout(width='300px'), **W)

uniprot_w = widgets.Text(
    value='', description='UniProt accession:',
    placeholder='e.g. Q26966',
    layout=widgets.Layout(width='320px'), **W)

sasa_threshold_w = widgets.FloatSlider(
    value=0.15, min=0.05, max=0.50, step=0.01,
    description='SASA threshold:',
    readout_format='.2f',
    layout=widgets.Layout(width='420px'), **W)

confirm_btn = widgets.Button(
    description='Confirm', button_style='success',
    layout=widgets.Layout(width='160px', height='34px'))
conf_out = widgets.Output()

def on_confirm(_):
    global TARGET_NAME, TARGET_LABEL, UNIPROT_ACCESSION, SASA_THRESHOLD
    TARGET_NAME        = target_name_w.value.strip()
    TARGET_LABEL       = target_label_w.value.strip()
    UNIPROT_ACCESSION  = uniprot_w.value.strip()
    SASA_THRESHOLD     = sasa_threshold_w.value
    with conf_out:
        clear_output()
        print(f'Configuration confirmed.')
        print(f'  Target     : {TARGET_NAME}  (label: {TARGET_LABEL})')
        print(f'  UniProt    : {UNIPROT_ACCESSION}')
        print(f'  SASA threshold: {SASA_THRESHOLD}')

confirm_btn.on_click(on_confirm)

# Initialise with defaults
TARGET_NAME       = target_name_w.value
TARGET_LABEL      = target_label_w.value
UNIPROT_ACCESSION = uniprot_w.value
SASA_THRESHOLD    = sasa_threshold_w.value

display(widgets.VBox([
    widgets.HTML('<h3>Notebook 00a — Configuration</h3>'),
    target_name_w, target_label_w, uniprot_w, sasa_threshold_w,
    widgets.HTML('<hr>'),
    widgets.HTML(
        '<i>Seed residues and sphere radius are defined in Cell 5b '
        'after structures are loaded.</i>'),
    confirm_btn, conf_out
]))


In [3]:
# Cell 3: Select output directory and PDB files

out = widgets.Output()
display(out)

# Output directory
with out:
    print('Select output directory...')
output_dir_str = ps_browse_folder('Select output directory for Notebook 00a results')
with out:
    if output_dir_str:
        output_dir = Path(output_dir_str)
    else:
        output_dir = Path.home() / 'BasementAntibodyWorks' / 'data' / TARGET_LABEL
        print(f'Using default: {output_dir}')
output_dir.mkdir(parents=True, exist_ok=True)
with out:
    print(f'Output dir: {output_dir}')

# PDB files — multi-select
with out:
    print()
    print('Select PDB files (hold Ctrl to select multiple)...')
pdb_paths = ps_browse_files_multi(
    title=f'Select PDB structures for {TARGET_NAME} (Ctrl = multi-select)')
with out:
    if not pdb_paths:
        raise FileNotFoundError('No PDB files selected. Re-run this cell.')
    print(f'Selected {len(pdb_paths)} structure(s):')
    for p in pdb_paths:
        print(f'  {Path(p).name}')


Output()

In [4]:
# Cell 4: UniProt fetch — DISABLED
#
# Q26966 is no longer used in this pipeline.
# All residue numbering uses PDB residue numbers throughout.
# The mutation filter in Cell 6 aligns PDB sequences to the
# REFERENCE structure (Cell 5b) to detect engineered mutations.
# No canonical sequence offset, no conversion needed.
#
# This cell is kept as a placeholder so downstream cell indices
# are not affected. Nothing runs here.

print("Cell 4: UniProt fetch disabled.")
print("Pipeline uses PDB residue numbering throughout.")
print("Alignment is PDB-to-PDB (reference structure), not PDB-to-UniProt.")


Cell 4: UniProt fetch disabled.
Pipeline uses PDB residue numbering throughout.
Alignment is PDB-to-PDB (reference structure), not PDB-to-UniProt.


In [5]:
# Cell 5: Load all PDB structures
#
# Extracts a sequence from chain 'A' by default (the common case), but this
# is not hardcoded — if a structure has no chain 'A' its first available
# chain is used instead. The full list of chains is kept per structure so
# Cell 5b can offer any chain as the reference chain.

parser = PDBParser(QUIET=True)
structures = {}

DEFAULT_CHAIN = 'A'

print(f'Loading {len(pdb_paths)} structure(s)...')
print()

for pdb_path in pdb_paths:
    stem = Path(pdb_path).stem.upper()
    if Path(pdb_path).stat().st_size < 1000:
        print(f'  SKIPPING {stem}: empty file ({Path(pdb_path).stat().st_size} bytes)')
        print(f'  Re-download: wget https://files.rcsb.org/download/{stem}.pdb -O {pdb_path}')
        continue

    struct = parser.get_structure(stem, pdb_path)
    model  = struct[0]

    available_chains = [c.get_id() for c in model]
    if not available_chains:
        print(f'WARNING: no chains found in {stem}. Skipping.')
        continue

    chain_id = DEFAULT_CHAIN if DEFAULT_CHAIN in available_chains else available_chains[0]
    chain = model[chain_id]
    protein_res = [r for r in chain if r.get_id()[0] == ' ']

    structures[stem] = {
        'path':             pdb_path,
        'structure':        struct,
        'model':            model,
        'chain':            chain,
        'chain_id':         chain_id,
        'available_chains': available_chains,
        'n_res':            len(protein_res),
    }

    seq = ''
    for r in protein_res:
        try:
            seq += seq1(r.get_resname())
        except Exception:
            seq += 'X'

    first = protein_res[0].get_id()[1] if protein_res else '?'
    last  = protein_res[-1].get_id()[1] if protein_res else '?'
    print(f'  {stem}: chain {chain_id}  {len(protein_res)} residues ({first}–{last})  '
          f'seq[:20]={seq[:20]}  chains={available_chains}')
    structures[stem]['pdb_seq']  = seq
    structures[stem]['res_nums'] = [r.get_id()[1] for r in protein_res]

print()
print(f'Successfully loaded: {list(structures.keys())}')


Loading 13 structure(s)...

  1MR5: chain A  621 residues (4–633)  seq[:20]=GSSRVELFKRQSSKVPFEKD  chains=['A']
  1MS0: chain A  623 residues (1–632)  seq[:20]=LAPGSSRVELFKRQSSKVPF  chains=['A', 'B', 'C', 'D']
  1MS1: chain A  622 residues (1–632)  seq[:20]=LAPGSSRVELFKRQSSKVPF  chains=['A', 'B']
  1MS3: chain A  623 residues (1–632)  seq[:20]=LAPGSSRVELFKRQSSKVPF  chains=['A', 'B']
  1MS4: chain A  623 residues (1–632)  seq[:20]=LAPGSSRVELFKRQSSKVPF  chains=['A', 'B']
  1MS5: chain A  623 residues (2–633)  seq[:20]=APGSSRVELFKRQSSKVPFE  chains=['A', 'B']
  1MS8: chain A  624 residues (1–633)  seq[:20]=LAPGSSRVELFKRQSSKVPF  chains=['A', 'B']
  1MS9: chain A  622 residues (2–632)  seq[:20]=APGSSRVELFKRQSSKVPFE  chains=['A', 'B', 'C', 'D']
  1S0I: chain A  623 residues (1–632)  seq[:20]=LAPGSSRVELFKRQSSKVPF  chains=['A', 'B']
  1S0J: chain A  623 residues (1–632)  seq[:20]=LAPGSSRVELFKRQSSKVPF  chains=['A']
  3B69: chain A  626 residues (-1–633)  seq[:20]=ASLAPGSSRVELFKRQSSKV  chains=['A'

In [6]:
# Cell 5b: Reference structure and region of interest selection
#
# Pick a reference structure and chain, mark a few "seed" residues you know
# are in the functional region, and set a sphere radius. Every residue
# within that sphere (in the reference structure) becomes the region that
# gets analysed across ALL loaded structures from Cell 6b onward.

def _find_region(ref_stem, ref_chain_id, seed_res_nums, sphere_radius):
    """Return (seed_centre, region_res_nums) for the given reference chain."""
    if ref_stem not in structures:
        print(f'WARNING: reference structure {ref_stem} not loaded.')
        return None, []

    model = structures[ref_stem]['structure'][0]
    if ref_chain_id not in [c.get_id() for c in model]:
        print(f'WARNING: chain {ref_chain_id} not found in {ref_stem}. '
              f'Available chains: {[c.get_id() for c in model]}')
        return None, []
    chain = model[ref_chain_id]

    if not seed_res_nums:
        print('No seed residues set yet. Click "Enter seed residues" above.')
        return None, []

    seed_coords = []
    for rn in seed_res_nums:
        try:
            res = chain[(' ', rn, ' ')]
            seed_coords.append(res['CA'].get_vector().get_array())
        except KeyError:
            print(f'WARNING: seed residue {rn} not found in chain {ref_chain_id} of {ref_stem}')

    if not seed_coords:
        print('No seed residues found in the reference chain. Check residue numbers.')
        return None, []

    centre = np.mean(seed_coords, axis=0)

    region = []
    for r in chain:
        if r.get_id()[0] != ' ':
            continue
        if 'CA' not in r:
            continue
        coord = r['CA'].get_vector().get_array()
        if np.linalg.norm(coord - centre) <= sphere_radius:
            region.append(r.get_id()[1])

    return centre, region


# ── Widgets ──────────────────────────────────────────────────────────────
ref_structure_w = widgets.Dropdown(
    options=list(structures.keys()),
    description='Reference structure (used to define the region):',
    style={'description_width': '320px'},
    layout=widgets.Layout(width='560px'))

ref_chain_w = widgets.Text(
    value='A', description='Chain ID:',
    layout=widgets.Layout(width='220px'), style={'description_width': '160px'})

seed_btn = widgets.Button(
    description='Enter seed residues', button_style='info',
    layout=widgets.Layout(width='200px', height='34px'))
seed_out = widgets.Output()

sphere_radius_w = widgets.FloatSlider(
    value=18.0, min=5.0, max=35.0, step=0.5,
    description='Sphere radius (A):', readout_format='.1f',
    layout=widgets.Layout(width='420px'), style={'description_width': '160px'})

confirm_region_btn = widgets.Button(
    description='Confirm', button_style='success',
    layout=widgets.Layout(width='160px', height='34px'))
region_out = widgets.Output()

SEED_RESIDUES = []

def on_seed_click(_):
    global SEED_RESIDUES
    script = """
Add-Type -AssemblyName Microsoft.VisualBasic
[Microsoft.VisualBasic.Interaction]::InputBox(
    "Enter seed residue numbers (comma-separated) that locate your region of interest.`nThese are residues you know are in the functional region.`nExample: 311,119,312",
    "Seed residues",
    ""
)
"""
    r = subprocess.run(["powershell.exe", "-NoProfile", "-Command", script],
                       capture_output=True, text=True)
    raw = r.stdout.strip()
    seeds = []
    for part in raw.split(','):
        part = part.strip()
        if not part:
            continue
        try:
            seeds.append(int(part))
        except ValueError:
            pass
    SEED_RESIDUES = seeds
    with seed_out:
        clear_output()
        print(f'Seed residues: {SEED_RESIDUES}' if seeds else 'No seed residues entered.')

seed_btn.on_click(on_seed_click)

def on_confirm_region(_):
    global REF_STRUCTURE, REF_CHAIN, SPHERE_RADIUS, seed_centre, region_res_nums
    REF_STRUCTURE = ref_structure_w.value
    REF_CHAIN     = ref_chain_w.value.strip().upper() or 'A'
    SPHERE_RADIUS = sphere_radius_w.value

    seed_centre, region_res_nums = _find_region(
        REF_STRUCTURE, REF_CHAIN, SEED_RESIDUES, SPHERE_RADIUS)

    with region_out:
        clear_output()
        print('Region of interest confirmed.')
        print(f'  Reference structure : {REF_STRUCTURE}')
        print(f'  Chain               : {REF_CHAIN}')
        print(f'  Seed residues       : {SEED_RESIDUES}')
        print(f'  Sphere radius       : {SPHERE_RADIUS} A')
        print(f'  Residues in region  : {len(region_res_nums)}')

confirm_region_btn.on_click(on_confirm_region)

display(widgets.VBox([
    widgets.HTML('<h3>Notebook 00a — Region of interest</h3>'),
    ref_structure_w, ref_chain_w,
    widgets.HBox([seed_btn, seed_out]),
    sphere_radius_w,
    widgets.HTML('<hr>'), confirm_region_btn, region_out
]))

# Initialise with defaults so downstream cells don't crash if Confirm
# has not been clicked yet in this session.
REF_STRUCTURE = ref_structure_w.value
REF_CHAIN     = ref_chain_w.value.strip().upper() or 'A'
SPHERE_RADIUS = sphere_radius_w.value
seed_centre, region_res_nums = _find_region(
    REF_STRUCTURE, REF_CHAIN, SEED_RESIDUES, SPHERE_RADIUS)


No seed residues set yet. Click "Enter seed residues" above.


In [7]:
# Cell 6: Align each PDB sequence to the reference structure sequence
#
# PURPOSE: detect engineered mutations between crystal structures.
# All numbering stays in PDB space throughout — no UniProt offset.
#
# Each structure's sequence is aligned to the REFERENCE structure
# (chosen in Cell 5b). Residues that differ from the reference
# are flagged as mutations and excluded from SASA averaging in Cell 8.
#
# The reference structure itself maps trivially: pdb_res_num -> pdb_res_num.
# All other structures are aligned to find their equivalent PDB positions.
#
# SCORING: mismatch_score must be more negative than open_gap_score
# so the aligner opens a gap rather than accepting a mismatch at
# positions where a construct starts/ends differently from the reference.
# Values: match=3, mismatch=-5, gap_open=-4, gap_extend=-0.5

from Bio.Align import PairwiseAligner
from Bio.PDB.Polypeptide import protein_letters_3to1

aligner = PairwiseAligner()
aligner.mode             = 'global'
aligner.match_score      =  3
aligner.mismatch_score   = -5    # more negative than gap_open to prefer gaps over mismatches
aligner.open_gap_score   = -4
aligner.extend_gap_score = -0.5

def align_to_reference(query_seq, query_res_nums, ref_seq, ref_res_nums):
    """
    Align query PDB sequence to reference PDB sequence.
    Returns dict: query_pdb_res_num -> ref_pdb_res_num
    Both sequences use PDB residue numbers.
    Only aligned (non-gap) positions are returned.
    """
    alignments = aligner.align(ref_seq, query_seq)
    best = next(iter(alignments))
    target_ranges, query_ranges = best.aligned

    mapping = {}  # query_pdb_res_num -> ref_pdb_res_num
    for (t_start, t_end), (q_start, q_end) in zip(target_ranges, query_ranges):
        block_len = t_end - t_start
        for i in range(block_len):
            ref_idx   = t_start + i
            query_idx = q_start + i
            if ref_idx < len(ref_res_nums) and query_idx < len(query_res_nums):
                mapping[query_res_nums[query_idx]] = ref_res_nums[ref_idx]
    return mapping

print('SEQUENCE ALIGNMENT — all structures vs reference')
print('=' * 65)
print(f'Reference structure: {REF_STRUCTURE}  chain {REF_CHAIN}')
print()

# Get reference sequence and residue numbers
ref_info     = structures[REF_STRUCTURE]
ref_seq      = ref_info['pdb_seq']
ref_res_nums = ref_info['res_nums']

# Store the reference mapping for each structure:
# structures[stem]['ref_mapping'] = {pdb_res_num -> ref_pdb_res_num}
# For the reference itself, mapping is identity
structures[REF_STRUCTURE]['ref_mapping'] = {rn: rn for rn in ref_res_nums}
structures[REF_STRUCTURE]['uniprot_mapping'] = {rn: rn for rn in ref_res_nums}

for stem, info in structures.items():
    if stem == REF_STRUCTURE:
        matches = len(ref_res_nums)
        pct     = 100.0
        print(f'  {stem}: {matches} residues mapped  |  sequence identity: {pct:.1f}%  [REFERENCE]')
        continue

    mapping = align_to_reference(
        info['pdb_seq'], info['res_nums'],
        ref_seq, ref_res_nums
    )
    structures[stem]['ref_mapping']    = mapping
    structures[stem]['uniprot_mapping'] = mapping  # kept for Cell 8 compatibility

    # Identity: fraction of mapped positions where amino acids match
    matches = 0
    mismatches = []
    for q_rn, r_rn in mapping.items():
        if q_rn not in info['res_nums']:
            continue
        q_idx = info['res_nums'].index(q_rn)
        q_aa  = info['pdb_seq'][q_idx]

        if r_rn not in ref_res_nums:
            continue
        r_idx = ref_res_nums.index(r_rn)
        r_aa  = ref_seq[r_idx]

        if q_aa == r_aa:
            matches += 1
        else:
            mismatches.append(f'{r_aa}{r_rn}→{q_aa}')

    pct = 100 * matches / max(len(mapping), 1)
    print(f'  {stem}: {len(mapping)} residues mapped  |  sequence identity: {pct:.1f}%')
    if mismatches:
        print(f'    Mutations vs {REF_STRUCTURE}: {", ".join(mismatches[:10])}')
        if len(mismatches) > 10:
            print(f'    ... and {len(mismatches)-10} more')

print()
print('Alignment complete. All residue numbers are PDB residue numbers.')
print(f'No UniProt offset. Tyr342 = 342, Tyr119 = 119, Trp312 = 312.')

# ── Verification ──────────────────────────────────────────────────────────────
print()
print('ALIGNMENT VERIFICATION — known TS anchor residues in reference')
print('-' * 65)

ANCHOR_CHECKS = [
    (342, 'Y', 'Tyr342 catalytic'),
    (119, 'Y', 'Tyr119 active site entrance'),
    (312, 'W', 'Trp312 active site entrance'),
]

all_ok = True
for pdb_rn, expected_aa, label in ANCHOR_CHECKS:
    if pdb_rn in ref_res_nums:
        idx    = ref_res_nums.index(pdb_rn)
        actual = ref_seq[idx]
        ok     = actual == expected_aa
        status = 'OK  ' if ok else 'FAIL'
        print(f'  {status}  {label}: PDB {pdb_rn} = {actual} (expected {expected_aa})')
        if not ok:
            all_ok = False
    else:
        print(f'  FAIL  {label}: PDB {pdb_rn} not found in reference')
        all_ok = False

print()
if all_ok:
    print('All anchor residues verified. PDB numbering is consistent.')
else:
    print('WARNING: anchor residue check failed. Verify reference structure.')
    raise RuntimeError('Alignment verification failed.')


SEQUENCE ALIGNMENT — all structures vs reference
Reference structure: 1MR5  chain A

  1MR5: 621 residues mapped  |  sequence identity: 100.0%  [REFERENCE]
  1MS0: 620 residues mapped  |  sequence identity: 100.0%
  1MS1: 619 residues mapped  |  sequence identity: 100.0%
  1MS3: 620 residues mapped  |  sequence identity: 100.0%
  1MS4: 620 residues mapped  |  sequence identity: 100.0%
  1MS5: 621 residues mapped  |  sequence identity: 99.8%
    Mutations vs 1MR5: H476→F
  1MS8: 621 residues mapped  |  sequence identity: 100.0%
  1MS9: 620 residues mapped  |  sequence identity: 100.0%
  1S0I: 620 residues mapped  |  sequence identity: 99.8%
    Mutations vs 1MR5: D59→A
  1S0J: 620 residues mapped  |  sequence identity: 99.8%
    Mutations vs 1MR5: D59→A
  3B69: 621 residues mapped  |  sequence identity: 100.0%
  3OPZ: 621 residues mapped  |  sequence identity: 100.0%

Alignment complete. All residue numbers are PDB residue numbers.
No UniProt offset. Tyr342 = 342, Tyr119 = 119, Trp312 =

In [ ]:
# Cell 7: Compute per-residue SASA (Shrake-Rupley) for every loaded structure
#
# Runs Shrake-Rupley on the full model (all chains) so that inter-chain
# contacts correctly occlude surface area, then records the per-residue
# SASA for the chain selected in Cell 5.
#
# structures[stem]['sasa_per_res'] = {pdb_res_num -> raw SASA in A^2}
# All numbers are PDB residue numbers. No UniProt offset.

sasa_calculator = ShrakeRupley()

print('Computing Shrake-Rupley SASA per residue...')
print()

for stem, info in structures.items():
    sasa_calculator.compute(info['model'], level='R')

    sasa_map = {}
    for res in info['chain']:
        if res.get_id()[0] != ' ':
            continue
        sasa_map[res.get_id()[1]] = res.sasa

    structures[stem]['sasa_per_res'] = sasa_map
    print(f'  {stem}: SASA computed for {len(sasa_map)} residues')

print()
print('SASA computation complete for all structures.')


In [12]:
# Cell 6c: Seed residue selection and region of interest definition
#
# Enter seed residue numbers in PDB numbering as they appear in the
# literature and PDB files. These numbers are used directly throughout
# the pipeline — no conversion, no offset.
#
# For trans-sialidase: 342 (catalytic Tyr), 119 (entrance Tyr), 312 (entrance Trp)
# These match Buschiazzo et al. 2002 and all subsequent TS literature.

# ── Step 1: enter seed residues via popup ──────────────────────────────────
_seed_script = """
Add-Type -AssemblyName Microsoft.VisualBasic
[Microsoft.VisualBasic.Interaction]::InputBox(
    "Enter seed residue numbers in PDB numbering (comma-separated).`n`nThese are the residue numbers from the literature and PDB files.`nThey are used directly throughout the pipeline.`n`nFor trans-sialidase: 342,119,312`n(Tyr342 catalytic, Tyr119 entrance, Trp312 entrance)",
    "Seed residues — PDB numbering",
    ""
)
"""
_seed_result = subprocess.run(
    ["powershell.exe", "-NoProfile", "-Command", _seed_script],
    capture_output=True, text=True
)
_seed_raw = _seed_result.stdout.strip()

pdb_seed_nums = []
for part in _seed_raw.split(','):
    part = part.strip()
    if not part:
        continue
    try:
        pdb_seed_nums.append(int(part))
    except ValueError:
        print(f'WARNING: could not parse "{part}" as integer — skipped')

if not pdb_seed_nums:
    raise ValueError('No seed residues entered. Re-run this cell.')

# ── Step 2: sphere radius ──────────────────────────────────────────────────
_radius_script = """
Add-Type -AssemblyName Microsoft.VisualBasic
[Microsoft.VisualBasic.Interaction]::InputBox(
    "Enter sphere radius in Angstroms.`n`nAll residues within this distance of the seed centre`nwill be included in the candidate pool analysis.`n`nRecommended: 18.0 A for an antibody epitope region.",
    "Sphere radius (Angstroms)",
    "18.0"
)
"""
_radius_result = subprocess.run(
    ["powershell.exe", "-NoProfile", "-Command", _radius_script],
    capture_output=True, text=True
)
try:
    SPHERE_RADIUS = float(_radius_result.stdout.strip())
except (ValueError, AttributeError):
    SPHERE_RADIUS = 18.0
    print(f'Could not parse radius — using default {SPHERE_RADIUS} A')

# ── Step 3: SEED_RESIDUES = PDB numbers directly ───────────────────────────
# No conversion. PDB numbers are the truth.
SEED_RESIDUES = pdb_seed_nums

print(f'SEED RESIDUES — PDB numbering')
print('=' * 65)
print(f'Reference structure : {REF_STRUCTURE}  chain {REF_CHAIN}')
print(f'Sphere radius       : {SPHERE_RADIUS} A')
print()

ref_info = structures[REF_STRUCTURE]
all_found = True
for pdb_rn in pdb_seed_nums:
    if pdb_rn in ref_info['res_nums']:
        idx    = ref_info['res_nums'].index(pdb_rn)
        pdb_aa = ref_info['pdb_seq'][idx]
        print(f'  PDB {pdb_rn:>4} = {pdb_aa}')
    else:
        print(f'  WARNING: PDB {pdb_rn} not found in reference structure')
        all_found = False

print()
if not all_found:
    raise ValueError('One or more seed residues not found. Check PDB residue numbers.')

# ── Step 4: sphere search in PDB coordinate space ──────────────────────────
ref_model     = structures[REF_STRUCTURE]['structure'][0]
ref_chain_obj = ref_model[REF_CHAIN]

seed_coords = []
for pdb_rn in pdb_seed_nums:
    try:
        res = ref_chain_obj[(' ', pdb_rn, ' ')]
        seed_coords.append(res['CA'].get_vector().get_array())
    except KeyError:
        print(f'WARNING: PDB {pdb_rn} not found in {REF_STRUCTURE} chain {REF_CHAIN}')

if not seed_coords:
    raise ValueError('No seed coordinates found.')

seed_centre = np.mean(seed_coords, axis=0)

region_res_nums = []
for r in ref_chain_obj:
    if r.get_id()[0] != ' ' or 'CA' not in r:
        continue
    coord = r['CA'].get_vector().get_array()
    if np.linalg.norm(coord - seed_centre) <= SPHERE_RADIUS:
        region_res_nums.append(r.get_id()[1])

# region_canonical = PDB residue numbers (no offset)
region_canonical = set(region_res_nums)

print(f'REGION OF INTEREST')
print('-' * 65)
print(f'Seed centre (Å) : ({seed_centre[0]:.2f}, {seed_centre[1]:.2f}, {seed_centre[2]:.2f})')
print(f'PDB residues in sphere: {len(region_res_nums)}')
print(f'Seed residues   : {SEED_RESIDUES}')
print(f'Sphere radius   : {SPHERE_RADIUS} A')
print()
print('All numbers are PDB residue numbers.')
print('Tyr342=342, Tyr119=119, Trp312=312 throughout.')


SEED RESIDUES — PDB numbering
Reference structure : 1MR5  chain A
Sphere radius       : 18.0 A

  PDB  342 = Y
  PDB  119 = Y
  PDB  312 = W

REGION OF INTEREST
-----------------------------------------------------------------
Seed centre (Å) : (23.26, 12.77, 29.67)
PDB residues in sphere: 130
Seed residues   : [342, 119, 312]
Sphere radius   : 18.0 A

All numbers are PDB residue numbers.
Tyr342=342, Tyr119=119, Trp312=312 throughout.


In [8]:
# Cell 6b: Verify region is populated
#
# region_res_nums: PDB residue numbers in the reference structure sphere.
# region_canonical: same values — PDB residue numbers.
# No UniProt conversion. No offset. What you see is what you get.

print(f'REGION CHECK — {TARGET_NAME}')
print('=' * 60)

if not region_res_nums:
    raise RuntimeError(
        'region_res_nums is empty. '
        'Run Cell 6c (seed residue selection) before this cell.'
    )

# region_canonical is kept as an alias for compatibility with downstream cells
region_canonical = set(region_res_nums)

print(f'Reference structure   : {REF_STRUCTURE}  chain {REF_CHAIN}')
print(f'Seed residues (PDB)   : {sorted(SEED_RESIDUES)}')
print(f'Sphere radius         : {SPHERE_RADIUS} A')
print(f'PDB residues in region: {len(region_res_nums)}')
print(f'  {sorted(region_res_nums)}')
print()
print('All residue numbers are PDB residue numbers throughout.')
print('Tyr342=342, Tyr119=119, Trp312=312 — consistent with all published literature.')


REGION CHECK — trans sialidase
Reference structure   : 1MR5  chain A
Seed residues (PDB)   : [119, 312, 342]
Sphere radius         : 18.0 A
PDB residues in region: 130
  [11, 13, 32, 33, 34, 35, 36, 37, 38, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 91, 92, 93, 94, 95, 96, 97, 98, 110, 111, 112, 113, 116, 117, 118, 119, 120, 121, 122, 123, 174, 175, 176, 177, 178, 179, 180, 181, 193, 194, 195, 196, 197, 201, 202, 203, 204, 205, 227, 228, 229, 230, 231, 232, 243, 244, 245, 246, 247, 248, 249, 250, 251, 274, 275, 276, 277, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 334, 338, 339, 340, 341, 342, 343, 344, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367]

All residue numbers are PDB residue numbers throughout.
Tyr342=342, Tyr119=119, Trp312=312 — consistent with all published literature.


In [14]:
# Cell 8: Map SASA values to PDB residue positions
#
# For each structure, collect SASA values per PDB residue number.
# MUTATION FILTER: if a residue differs from the same position in the
# reference structure (REF_STRUCTURE), exclude it from the SASA average
# at that position. This removes engineered mutations (e.g. Y342H in 3PJQ)
# without discarding the whole structure.
#
# All numbers are PDB residue numbers. No UniProt offset.

# sasa_collection[pdb_res_num] = list of (stem, sasa, aa)
sasa_collection = defaultdict(list)

mutation_exclusions = []
total_processed  = 0
total_excluded   = 0

ref_seq      = structures[REF_STRUCTURE]['pdb_seq']
ref_res_nums = structures[REF_STRUCTURE]['res_nums']

for stem, info in structures.items():

    # Safety checks — skip structures that did not complete Cell 6 or Cell 7
    if 'sasa_per_res' not in info:
        print(f'WARNING: {stem} has no SASA data (Cell 7 may not have run) — skipping')
        continue
    if 'ref_mapping' not in info:
        print(f'WARNING: {stem} has no alignment mapping (Cell 6 may not have run) — skipping')
        continue

    mapping  = info['ref_mapping']    # query_pdb_res_num -> ref_pdb_res_num
    sasa_map = info['sasa_per_res']   # pdb_res_num -> raw SASA

    for q_rn, r_rn in mapping.items():
        if q_rn not in sasa_map:
            continue

        total_processed += 1

        # Get amino acid in this structure at this PDB position
        if q_rn not in info['res_nums']:
            continue
        q_idx = info['res_nums'].index(q_rn)
        q_aa  = info['pdb_seq'][q_idx]

        # Get amino acid in reference structure at equivalent PDB position
        if r_rn not in ref_res_nums:
            continue
        r_idx  = ref_res_nums.index(r_rn)
        ref_aa = ref_seq[r_idx]

        # MUTATION FILTER
        # Exclude residues that differ from the reference structure.
        # Catches all mutations regardless of chemical similarity —
        # conservative substitutions (e.g. Tyr->Phe) are also excluded
        # because the SASA reflects the mutant, not the canonical residue.
        if q_aa != ref_aa:
            mutation_exclusions.append({
                'structure':   stem,
                'pdb_res_num': q_rn,
                'ref_res_num': r_rn,
                'query_aa':    q_aa,
                'ref_aa':      ref_aa,
            })
            total_excluded += 1
            continue

        # Include this SASA value — keyed by REFERENCE PDB residue number
        sasa_collection[r_rn].append({
            'structure':   stem,
            'pdb_res_num': q_rn,
            'aa':          q_aa,
            'raw_sasa':    sasa_map[q_rn],
        })

# ── Report ────────────────────────────────────────────────────────────────────
print('MUTATION FILTER REPORT')
print('=' * 60)
print(f'Total residues processed : {total_processed}')
print(f'Mutant residues excluded : {total_excluded}')
print(f'Positions with data      : {len(sasa_collection)}')
print()

if mutation_exclusions:
    excl_df = pd.DataFrame(mutation_exclusions)
    print('Excluded mutant residues by structure:')
    for stem, grp in excl_df.groupby('structure'):
        muts = [
            f'{r["ref_aa"]}{int(r["ref_res_num"])}{r["query_aa"]}'
            for _, r in grp.iterrows()
        ]
        print(f'  {stem}: {", ".join(muts)}')
    print()
    print(f'Structures with mutations: {excl_df["structure"].unique().tolist()}')
else:
    print('No mutations detected vs reference structure.')

print()

# Normalise using global max SASA across included residues only
all_raw = [
    e['raw_sasa']
    for entries in sasa_collection.values()
    for e in entries
]
global_max_sasa = max(all_raw) if all_raw else 1.0
print(f'Global max SASA (mutations excluded): {global_max_sasa:.2f} A²')

# Alias for downstream compatibility
uniprot_sasa_collection = sasa_collection

MUTATION FILTER REPORT
Total residues processed : 0
Mutant residues excluded : 0
Positions with data      : 0

No mutations detected vs reference structure.

Global max SASA (mutations excluded): 1.00 A²


In [16]:
# Cell 9: Region-focused cross-structure SASA analysis
#
# Analyzes ONLY the residues in the region of interest (from Cell 6b)
# across ALL loaded structures. Records how each residue's exposure
# changes across structural states.
#
# All residue numbers are PDB residue numbers.
# Amino acid identity comes from the reference structure sequence —
# no UniProt sequence needed.

# Build a quick lookup: pdb_res_num -> single-letter AA from reference structure
ref_aa_lookup = {}
ref_info = structures[REF_STRUCTURE]
for i, rn in enumerate(ref_info['res_nums']):
    ref_aa_lookup[rn] = ref_info['pdb_seq'][i]

region_data = {}
for pdb_pos in sorted(region_canonical):

    # Get amino acid from reference structure
    pdb_aa = ref_aa_lookup.get(pdb_pos, '?')

    if pdb_pos not in uniprot_sasa_collection:
        region_data[pdb_pos] = {
            'pdb_res_num':    pdb_pos,
            'uniprot_residue': pdb_aa,   # field name kept for downstream compatibility
            'mean_sasa':       0.0,
            'std_sasa':        0.0,
            'min_sasa':        0.0,
            'max_sasa':        0.0,
            'n_structures':    0,
            'n_exposed':       0,
            'consistency':     0.0,
            'per_structure':   {},
            'category':        'missing',
        }
        continue

    entries    = uniprot_sasa_collection[pdb_pos]
    rel_values = [e['raw_sasa'] / global_max_sasa for e in entries]

    n_exposed   = sum(1 for v in rel_values if v >= SASA_THRESHOLD)
    consistency = n_exposed / max(len(rel_values), 1)
    mean_sasa   = float(np.mean(rel_values))
    std_sasa    = float(np.std(rel_values))
    min_sasa    = float(np.min(rel_values))
    max_sasa    = float(np.max(rel_values))
    per_structure = {
        e['structure']: round(e['raw_sasa'] / global_max_sasa, 4)
        for e in entries
    }

    # Classification
    if consistency >= 0.85 and min_sasa >= 0.10:
        category = 'A_consistently_exposed'
    elif consistency >= 0.60 and mean_sasa >= SASA_THRESHOLD:
        category = 'B_mostly_exposed'
    elif consistency >= 0.30:
        category = 'C_variable'
    else:
        category = 'D_mostly_buried'

    region_data[pdb_pos] = {
        'pdb_res_num':    pdb_pos,
        'uniprot_residue': pdb_aa,   # field name kept for downstream compatibility
        'mean_sasa':       mean_sasa,
        'std_sasa':        std_sasa,
        'min_sasa':        min_sasa,
        'max_sasa':        max_sasa,
        'n_structures':    len(entries),
        'n_exposed':       n_exposed,
        'consistency':     round(consistency, 3),
        'per_structure':   per_structure,
        'category':        category,
    }

region_df = pd.DataFrame.from_dict(region_data, orient='index')

# Print summary
print(f'REGION ANALYSIS — {TARGET_NAME}')
print(f'Reference structure : {REF_STRUCTURE} chain {REF_CHAIN}')
print(f'Seed residues (PDB) : {SEED_RESIDUES}')
print(f'Region size         : {len(region_data)} residues')
print(f'Structures analysed : {len(structures)}')
print()

for cat, label in [
    ('A_consistently_exposed', 'A — Consistently exposed (best antibody targets)'),
    ('B_mostly_exposed',       'B — Mostly exposed (good targets)'),
    ('C_variable',             'C — Variable (use with caution)'),
    ('D_mostly_buried',        'D — Mostly buried (avoid)'),
    ('missing',                'Missing — not found in enough structures'),
]:
    subset   = region_df[region_df['category'] == cat]
    residues = [
        f"{row['uniprot_residue']}{int(pos)}"
        for pos, row in subset.iterrows()
    ]
    print(f'{label}: {len(subset)}')
    if residues:
        print(f'  {residues}')

print()
display(region_df[[
    'uniprot_residue', 'mean_sasa', 'std_sasa',
    'min_sasa', 'max_sasa', 'consistency',
    'n_exposed', 'n_structures', 'category'
]].round(3))

REGION ANALYSIS — trans sialidase
Reference structure : 1MR5 chain A
Seed residues (PDB) : [342, 119, 312]
Region size         : 130 residues
Structures analysed : 12

A — Consistently exposed (best antibody targets): 0
B — Mostly exposed (good targets): 0
C — Variable (use with caution): 0
D — Mostly buried (avoid): 0
Missing — not found in enough structures: 130
  ['F11', 'R13', 'H32', 'S33', 'F34', 'R35', 'L36', 'P37', 'A38', 'I49', 'A50', 'D51', 'A52', 'R53', 'Y54', 'E55', 'T56', 'S57', 'F58', 'D59', 'N60', 'S61', 'L62', 'I63', 'D64', 'T65', 'V91', 'S92', 'R93', 'V94', 'V95', 'D96', 'P97', 'T98', 'V110', 'G111', 'S112', 'Y113', 'S116', 'R117', 'S118', 'Y119', 'W120', 'T121', 'S122', 'H123', 'Q174', 'F175', 'L176', 'G177', 'G178', 'A179', 'G180', 'V181', 'P193', 'V194', 'Q195', 'V196', 'T197', 'K201', 'Q202', 'V203', 'F204', 'S205', 'G227', 'C228', 'S229', 'E230', 'P231', 'V232', 'N243', 'T244', 'R245', 'V246', 'D247', 'Y248', 'R249', 'R250', 'R251', 'W274', 'G275', 'P276', 'S277', 

,uniprot_residue,mean_sasa,std_sasa,min_sasa,max_sasa,consistency,n_exposed,n_structures,category
11,F,0.0,0.0,0.0,0.0,0.0,0,0,missing
13,R,0.0,0.0,0.0,0.0,0.0,0,0,missing
32,H,0.0,0.0,0.0,0.0,0.0,0,0,missing
33,S,0.0,0.0,0.0,0.0,0.0,0,0,missing
34,F,0.0,0.0,0.0,0.0,0.0,0,0,missing
...,...,...,...,...,...,...,...,...,...
363,V,0.0,0.0,0.0,0.0,0.0,0,0,missing
364,Y,0.0,0.0,0.0,0.0,0.0,0,0,missing
365,S,0.0,0.0,0.0,0.0,0.0,0,0,missing
366,L,0.0,0.0,0.0,0.0,0.0,0,0,missing


In [17]:
# Cell 10: Consistently exposed residues — the candidate pool for Notebook 01
#
# Category A residues are exposed in at least 85% of structures and never
# fully buried. These are the most reliable antibody targets.

cat_A = region_df[region_df['category'] == 'A_consistently_exposed'].sort_values(
    'mean_sasa', ascending=False)
cat_B = region_df[region_df['category'] == 'B_mostly_exposed'].sort_values(
    'mean_sasa', ascending=False)

print('CONSISTENTLY EXPOSED RESIDUES (Category A)')
print('These residues are reliably accessible regardless of protein state.')
print('Recommended as the candidate pool for Notebook 01 and the Bayesian loop.')
print('=' * 70)
if len(cat_A) > 0:
    for pos, row in cat_A.iterrows():
        structs_exposed = [s for s, v in row['per_structure'].items() 
                          if v >= SASA_THRESHOLD]
        print(f"  {row['uniprot_residue']}{int(pos):4d}  "
              f"mean={row['mean_sasa']:.3f}  std={row['std_sasa']:.3f}  "
              f"min={row['min_sasa']:.3f}  "
              f"exposed in {row['n_exposed']}/{row['n_structures']} structures")
else:
    print('  No Category A residues found. Consider lowering SASA_THRESHOLD')
    print('  or adjusting the consistency cutoff in Cell 9.')

print()
print('MOSTLY EXPOSED RESIDUES (Category B)')
print('Exposed in most but not all structures.')
if len(cat_B) > 0:
    for pos, row in cat_B.iterrows():
        print(f"  {row['uniprot_residue']}{int(pos):4d}  "
              f"mean={row['mean_sasa']:.3f}  std={row['std_sasa']:.3f}  "
              f"exposed in {row['n_exposed']}/{row['n_structures']} structures")

candidate_pool_00a = sorted([int(pos) for pos in cat_A.index])
print()
print(f'Candidate pool (Category A only): {candidate_pool_00a}')


CONSISTENTLY EXPOSED RESIDUES (Category A)
These residues are reliably accessible regardless of protein state.
Recommended as the candidate pool for Notebook 01 and the Bayesian loop.
  No Category A residues found. Consider lowering SASA_THRESHOLD
  or adjusting the consistency cutoff in Cell 9.

MOSTLY EXPOSED RESIDUES (Category B)
Exposed in most but not all structures.

Candidate pool (Category A only): []


In [20]:
# Cell 10b: Paratope contact scoring — rank candidate pool by antibody-interaction favorability
#
# Empirical scoring table derived from:
#   Reis et al. (2022) Front. Mol. Biosci. — 1425 Ab-Ag structures (SAbDab)
#   Kodchakorn et al. (2026) Sci. Reports — DockQ structural determinants
#
# References:
#   Reis et al. 2022 — Figure 4 (aa composition), Figure 6D (hydrophobic cluster),
#                      Figure 10 (Tyr/Ser interactions), Table 1
#   Kodchakorn et al. 2026 — Figure 8 (hydrophobicity), secondary structure analysis

# ── Paratope base scores ────────────────────────────────────────────────────────
PARATOPE_BASE_SCORES = {
    'TYR': 1.00, 'TRP': 0.85, 'SER': 0.75, 'ASP': 0.70,
    'ARG': 0.68, 'PHE': 0.65, 'GLY': 0.55, 'THR': 0.50,
    'ASN': 0.50, 'ALA': 0.40, 'VAL': 0.30, 'ILE': 0.25,
    'LEU': 0.25, 'GLN': 0.15, 'LYS': 0.15, 'GLU': 0.15,
    'HIS': 0.10, 'MET': 0.10, 'PRO': 0.05, 'CYS': 0.05,
}

ONE_TO_THREE = {
    'A':'ALA','R':'ARG','N':'ASN','D':'ASP','C':'CYS',
    'Q':'GLN','E':'GLU','G':'GLY','H':'HIS','I':'ILE',
    'L':'LEU','K':'LYS','M':'MET','F':'PHE','P':'PRO',
    'S':'SER','T':'THR','W':'TRP','Y':'TYR','V':'VAL',
}

COIL_DSSP_CODES = {' ', 'T', 'S', '-', 'C'}

# ── Resolve candidate pool ──────────────────────────────────────────────────────
# candidate_pool_00a may be named differently depending on which cell ran.
# Try several fallbacks.
if 'candidate_pool_00a' in dir() and candidate_pool_00a:
    pool = candidate_pool_00a
elif 'candidate_pool_A' in dir() and candidate_pool_A:
    pool = candidate_pool_A
else:
    # Fall back to all Category A residues from region_data
    pool = [
        pos for pos, rd in region_data.items()
        if rd.get('category') == 'A_consistently_exposed'
    ]
    print(f'NOTE: candidate_pool_00a not found — using all Category A residues from region_data')

print(f'PARATOPE CONTACT SCORING — {TARGET_NAME}')
print('=' * 70)
print(f'Source: Reis et al. 2022 (1425 Ab-Ag structures), Kodchakorn et al. 2026')
print(f'Candidate pool: {sorted(pool)} ({len(pool)} residues)')
print()

if not pool:
    raise ValueError(
        'Candidate pool is empty. '
        'Check that Cell 10 ran successfully and produced candidate_pool_00a or candidate_pool_A.'
    )

# ── Compute contact scores ──────────────────────────────────────────────────────
contact_score_records = []

for pos in sorted(pool):
    if pos not in region_data:
        print(f'  WARNING: pos {pos} in candidate pool but not in region_data — skipping')
        continue

    rd         = region_data[pos]
    res_name_1 = rd.get('uniprot_residue', '?')
    res_name_3 = ONE_TO_THREE.get(res_name_1, 'UNK')

    base_score = PARATOPE_BASE_SCORES.get(res_name_3, 0.20)

    dssp_code  = rd.get('dssp', None)
    is_coil    = (dssp_code in COIL_DSSP_CODES) if dssp_code is not None else None
    coil_bonus = 0.30 if is_coil else 0.0

    mean_sasa  = rd.get('mean_sasa', 0.0)
    sasa_bonus = 0.20 if mean_sasa > 0.30 else 0.0

    raw_score     = base_score * (1 + coil_bonus + sasa_bonus)
    contact_score = min(round(raw_score, 4), 1.0)

    if base_score >= 0.75:
        tier = 'Tier 1 — strongly favoured'
    elif base_score >= 0.40:
        tier = 'Tier 2 — moderately favoured'
    elif base_score >= 0.20:
        tier = 'Tier 3 — neutral'
    else:
        tier = 'Penalised — depleted in paratopes'

    contact_score_records.append({
        'pdb_res_num':  pos,
        'residue_1':    res_name_1,
        'residue_3':    res_name_3,
        'base_score':   base_score,
        'mean_sasa':    round(mean_sasa, 4),
        'is_coil':      is_coil,
        'coil_bonus':   round(coil_bonus, 2),
        'sasa_bonus':   round(sasa_bonus, 2),
        'contact_score': contact_score,
        'tier':          tier,
    })

if not contact_score_records:
    raise ValueError(
        'No contact scores computed. '
        'Check that region_data contains entries for the candidate pool residues.'
    )

contact_score_df = pd.DataFrame(contact_score_records).sort_values(
    'contact_score', ascending=False
).reset_index(drop=True)
contact_score_df['rank'] = contact_score_df.index + 1

print(f'Candidate pool size : {len(contact_score_df)} residues')
print()
print('RANKED CANDIDATE POOL BY CONTACT SCORE')
print('-' * 70)
display(contact_score_df[[
    'rank', 'pdb_res_num', 'residue_3', 'base_score',
    'mean_sasa', 'is_coil', 'contact_score', 'tier'
]].to_string(index=False))

print()
print('TIER SUMMARY')
print('-' * 40)
for tier_label in [
    'Tier 1 — strongly favoured',
    'Tier 2 — moderately favoured',
    'Tier 3 — neutral',
    'Penalised — depleted in paratopes',
]:
    subset = contact_score_df[contact_score_df['tier'] == tier_label]
    if len(subset) > 0:
        names = [f"{r['residue_3']}{int(r['pdb_res_num'])}"
                 for _, r in subset.iterrows()]
        print(f'{tier_label} ({len(subset)}): {", ".join(names)}')

# Tier-1 aromatic anchors check
EXPECTED_TIER1 = [
    int(row['pdb_res_num'])
    for _, row in contact_score_df.iterrows()
    if row['residue_3'] in ('TYR', 'TRP') and row['contact_score'] >= 0.65
]

print()
if EXPECTED_TIER1:
    print(f'Tier-1 aromatic anchors (Tyr/Trp, score >= 0.65): {EXPECTED_TIER1}')
    print('These should be prioritised as anchor residues in the Bayesian controller.')
else:
    print('WARNING: No Tyr or Trp residues found in candidate pool with score >= 0.65.')
    print('Check that the candidate pool was correctly populated from Cell 10.')

# ── Contact scores dict for Cell 12 ───────────────────────────────────────────
contact_scores_dict = {
    str(int(row['pdb_res_num'])): row['contact_score']
    for _, row in contact_score_df.iterrows()
}

print()
print(f'contact_scores_dict populated: {len(contact_scores_dict)} entries')
print('Cell 12 will write this into the consensus SASA JSON for Notebook 00.')

NOTE: candidate_pool_00a not found — using all Category A residues from region_data
PARATOPE CONTACT SCORING — trans sialidase
Source: Reis et al. 2022 (1425 Ab-Ag structures), Kodchakorn et al. 2026
Candidate pool: [] (0 residues)



ValueError: Candidate pool is empty. Check that Cell 10 ran successfully and produced candidate_pool_00a or candidate_pool_A.

In [ ]:
# Cell 11: Visualisation
# 
# Three panels:
# Left: heatmap of SASA values — structures x residues (the transition matrix)
# Centre: mean SASA bar chart for the region coloured by category
# Right: consistency vs mean SASA scatter

import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

struct_names = list(structures.keys())
region_positions = sorted(region_data.keys())
residue_labels = [f"{region_data[p]['uniprot_residue']}{p}" for p in region_positions]

# Build heatmap matrix as a DataFrame (structures as columns, residues as index)
heatmap_data = np.zeros((len(region_positions), len(struct_names)))
for i, pos in enumerate(region_positions):
    for j, stem in enumerate(struct_names):
        heatmap_data[i, j] = region_data[pos]['per_structure'].get(stem, 0.0)

heatmap_data_df = pd.DataFrame(heatmap_data, index=residue_labels, columns=struct_names)

ax = axes[0]
sns.heatmap(
    heatmap_data_df,
    ax=ax,
    cmap='RdYlGn',
    vmin=0, vmax=0.8,
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Relative SASA', 'shrink': 0.7, 'pad': 0.02},
    xticklabels=True,
    yticklabels=True,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('SASA across structures\n(green=exposed, red=buried)',
             fontsize=14, fontweight='bold', pad=15)
cbar = ax.collections[0].colorbar
cbar.set_label('Relative SASA', fontsize=13, fontweight='bold')
cbar.ax.tick_params(labelsize=11)

# Centre panel: horizontal bar chart, hue = category
bar_df = pd.DataFrame({
    'residue':    residue_labels,
    'mean_sasa':  [region_data[p]['mean_sasa'] for p in region_positions],
    'std_sasa':   [region_data[p]['std_sasa'] for p in region_positions],
    'category':   [region_data[p]['category'] for p in region_positions],
})

ax2 = axes[1]
hue_order = [c for c in CATEGORY_SASA if c in bar_df['category'].unique()]
sns.barplot(
    data=bar_df, y='residue', x='mean_sasa', hue='category',
    order=residue_labels, hue_order=hue_order, palette=CATEGORY_SASA,
    dodge=False, edgecolor='white', linewidth=0.4, ax=ax2,
)
ax2.errorbar(
    bar_df['mean_sasa'], range(len(bar_df)),
    xerr=bar_df['std_sasa'], fmt='none', color='black', capsize=2, linewidth=0.7,
)
ax2.axvline(SASA_THRESHOLD, color='black', linestyle='--', linewidth=1.5,
            label=f'Threshold ({SASA_THRESHOLD})')
ax2.set_xlabel('Mean relative SASA', fontsize=13, fontweight='bold')
ax2.set_ylabel('')
ax2.tick_params(axis='y', labelsize=8)
ax2.tick_params(axis='x', labelsize=11)
ax2.set_title('Mean SASA ± std\n(colour = category)', fontsize=14, fontweight='bold', pad=15)

category_names = {
    'A_consistently_exposed': 'A: Consistently exposed',
    'B_mostly_exposed':       'B: Mostly exposed',
    'C_variable':             'C: Variable',
    'D_mostly_buried':        'D: Mostly buried',
    'missing':                'Missing',
}
handles, labels = ax2.get_legend_handles_labels()
clean_labels = [category_names.get(l, l) for l in labels]
ax2.legend(handles, clean_labels, fontsize=9, loc='lower right')

# Right panel: consistency vs mean SASA scatter
scatter_df = pd.DataFrame({
    'residue':      residue_labels,
    'mean_sasa':    [region_data[p]['mean_sasa'] for p in region_positions],
    'consistency':  [region_data[p]['consistency'] for p in region_positions],
    'category':     [region_data[p]['category'] for p in region_positions],
    'n_structures': [region_data[p]['n_structures'] for p in region_positions],
})

ax3 = axes[2]
sns.scatterplot(
    data=scatter_df, x='mean_sasa', y='consistency',
    hue='category', hue_order=hue_order, palette=CATEGORY_SASA,
    size='n_structures', sizes=(40, 120),
    edgecolor='white', linewidth=0.4, ax=ax3, zorder=3,
)
for _, row in scatter_df.iterrows():
    ax3.annotate(row['residue'], (row['mean_sasa'], row['consistency']),
                 fontsize=8, xytext=(4, 4), textcoords='offset points')

ax3.axvline(SASA_THRESHOLD, color='black', linestyle='--', linewidth=1, alpha=0.6)
ax3.axhline(0.85, color='green', linestyle='--', linewidth=1, alpha=0.6)

quad_kwargs = dict(fontsize=10, fontweight='bold', alpha=0.55, color='#555555',
                    transform=ax3.transAxes)
ax3.text(0.97, 0.95, 'Best targets',              ha='right', va='top',    **quad_kwargs)
ax3.text(0.03, 0.95, 'Accessible but variable',   ha='left',  va='top',    **quad_kwargs)
ax3.text(0.97, 0.05, 'Partially accessible',      ha='right', va='bottom', **quad_kwargs)
ax3.text(0.03, 0.05, 'Avoid',                     ha='left',  va='bottom', **quad_kwargs)

ax3.set_xlabel('Mean SASA', fontsize=13, fontweight='bold')
ax3.set_ylabel('Consistency (fraction exposed)', fontsize=13, fontweight='bold')
ax3.tick_params(labelsize=11)
ax3.set_title('Consistency vs mean SASA', fontsize=14, fontweight='bold', pad=15)
ax3.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.02, 0.5),
           title='Category / n structures')

plt.suptitle(
    f'{TARGET_NAME} — Region analysis across {len(structures)} structures\n'
    f'Seed: {SEED_RESIDUES}  Sphere: {SPHERE_RADIUS} Å  '
    f'Reference: {REF_STRUCTURE}',
    fontsize=16, fontweight='bold'
)
plt.tight_layout()

fig_path = output_dir / f'{TARGET_LABEL}_region_analysis.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
pass
print(f'Figure saved: {fig_path}')


In [ ]:
# Cell 12: Save outputs for Notebook 01

output = {
    'target_name':          TARGET_NAME,
    'target_label':         TARGET_LABEL,
    'reference_structure_used':    REF_STRUCTURE,
    'reference_structure':  REF_STRUCTURE,
    'reference_chain':      REF_CHAIN,
    'seed_residues':        SEED_RESIDUES,
    'sphere_radius':        SPHERE_RADIUS,
    'sasa_threshold':       SASA_THRESHOLD,
    'n_structures':         len(structures),
    'structures':           list(structures.keys()),
    'global_max_sasa':      global_max_sasa,
    'region_canonical':     sorted(list(region_canonical)),
    'candidate_pool_A':     candidate_pool_00a,
    'region_data':          {str(k): v for k, v in region_data.items()},
    'contact_scores':      contact_scores_dict,
}

out_path = output_dir / f'{TARGET_LABEL}_consensus_sasa.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'NOTEBOOK 00a COMPLETE — {TARGET_NAME}')
print('=' * 60)
print(f'Structures     : {list(structures.keys())}')
print(f'Region size    : {len(region_data)} residues')
print(f'Category A     : {len(cat_A)} consistently exposed')
print(f'Category B     : {len(cat_B)} mostly exposed')
print(f'Output JSON    : {out_path}')
print()
print('Category A residues (candidate pool for Notebook 01):')
print(f'  {candidate_pool_00a}')
